# IPSA (CC01-1940): EDA y preprocesamiento para clasificación

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/juanestebancg2806/sugarcane-yield-quality-prediction/blob/main/01_eda_ipsa.ipynb)

Este notebook prepara los datos de la **parte de clasificación** del taller. Usa `BD_IPSA_1940.xlsx`, un
subconjunto del Ingenio Providencia que el propio Ingenio construyó con estas restricciones:

- **Variedad:** CC01-1940
- **Maduración:** química (madurante BONUS 250 EC)
- **Cosecha:** mecanizada en verde

Cada fila corresponde a una cosecha de una suerte (hacienda `FAZ` + suerte `TAL` en un `periodo`).
Las conclusiones aplican solo a suertes con estas características y **no describen al Ingenio completo**.
Este análisis es independiente del de regresión sobre el histórico de suertes
(`01_eda_suertes.ipynb` / `02_modelos_regresion.ipynb`).

**Objetivo:** predecir, antes de la cosecha, el nivel (bajo / medio / alto) de dos targets que se tratan
como problemas independientes:

- **TCH:** toneladas de caña por hectárea (cantidad)
- **Sacarosa:** % de sacarosa en caña (calidad)

**Flujo del notebook**

0. Alcance del dataset
1. Diccionario de datos
2. Chequeo de *leakage*
3. Auditoría de nulos disfrazados
4. Selección de predictores
5. Análisis de los dos targets
6. Relación predictor → target
7. VIF
8. Definición de las clases (terciles vs. umbrales de negocio)
9. Exportación y limitaciones

**Siguiente paso:** los modelos (logística multinomial regularizada y KNN) están en
`02_modelos_clasificacion.ipynb`.

> **Datos:** `BD_IPSA_1940.xlsx` no está en el repositorio porque el enunciado prohíbe publicar los datos.
> En Colab hay que subirlo manualmente al directorio de trabajo antes de ejecutar.

In [11]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import statsmodels.api as sm
from scipy import stats



pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", None)

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 11

%matplotlib inline

## Carga de datos

Se lee la hoja `BD_IPSA` del archivo `BD_IPSA_1940.xlsx`, se elimina el índice residual que trae el
Excel y se revisa una vista general: primeras filas, tipos de dato, conteo de nulos y estadísticos
descriptivos.

In [12]:
# Lectura del dataset IPSA
df_ipsa = pd.read_excel("BD_IPSA_1940.xlsx", sheet_name="BD_IPSA")

# La primera columna de IPSA es un índice residual del Excel
if "Unnamed: 0" in df_ipsa.columns:
    df_ipsa = df_ipsa.drop(columns=["Unnamed: 0"])

print("BD_IPSA:", df_ipsa.shape)


BD_IPSA: (2187, 20)


In [13]:
# Vista rápida: IPSA (variedad CC01-1940)
display(df_ipsa.head())
df_ipsa.info()
display(df_ipsa.describe(include="all").T)

,NOME,FAZ,TAL,tipocorte,variedad,madurada,producto,dosismad,semsmad,edad,cortes,me,vejez,sacarosa,mes,periodo,TCH,lluvias,grupo_tenencia,pct_diatrea
0,AMAIME SILCA,81291,40,Mecanizado Verde,CC01-1940,SI,BONUS 250 EC REGULADOR FISIOLÓGICO,0.8,8.3,12.3,4,12.7,2.4,14.0,12,202012,112,137,3,6.2
1,AMAIME SILCA,81291,41,Mecanizado Verde,CC01-1940,SI,BONUS 250 EC REGULADOR FISIOLÓGICO,0.8,6.3,11.2,2,7.8,2.3,13.0,3,201903,157,0,3,3.5
2,AMAIME SILCA,81291,41,Mecanizado Verde,CC01-1940,SI,BONUS 250 EC REGULADOR FISIOLÓGICO,0.6,7.9,12.2,3,8.8,1.8,13.3,3,202003,167,68,3,4.3
3,AMAIME SILCA,81291,43,Mecanizado Verde,CC01-1940,SI,BONUS 250 EC REGULADOR FISIOLÓGICO,0.8,6.6,13.1,1,6.1,2.5,13.4,3,201903,156,0,3,3.5
4,AMAIME SILCA,81291,43,Mecanizado Verde,CC01-1940,SI,BONUS 250 EC REGULADOR FISIOLÓGICO,0.6,8.1,12.2,2,7.9,2.1,14.0,3,202003,151,68,3,4.3


<class 'pandas.DataFrame'>
RangeIndex: 2187 entries, 0 to 2186
Data columns (total 20 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   NOME            2187 non-null   str    
 1   FAZ             2187 non-null   int64  
 2   TAL             2187 non-null   object 
 3   tipocorte       2187 non-null   str    
 4   variedad        2187 non-null   str    
 5   madurada        2187 non-null   str    
 6   producto        2187 non-null   str    
 7   dosismad        2187 non-null   float64
 8   semsmad         2187 non-null   float64
 9   edad            2187 non-null   float64
 10  cortes          2187 non-null   int64  
 11  me              2187 non-null   float64
 12  vejez           2187 non-null   float64
 13  sacarosa        2187 non-null   float64
 14  mes             2187 non-null   int64  
 15  periodo         2187 non-null   int64  
 16  TCH             2187 non-null   int64  
 17  lluvias         2187 non-null   int64  
 18 

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
NOME,2187,285,SAN MIGUEL CARVAJAL,101,NaN,NaN,NaN,NaN,NaN,NaN,NaN
FAZ,2187.0,NaN,NaN,NaN,80588.332876,572.818299,80100.0,80222.0,80396.0,80660.0,82519.0
TAL,2187.0,273.0,1.0,258.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
tipocorte,2187,1,Mecanizado Verde,2187,NaN,NaN,NaN,NaN,NaN,NaN,NaN
variedad,2187,1,CC01-1940,2187,NaN,NaN,NaN,NaN,NaN,NaN,NaN
madurada,2187,1,SI,2187,NaN,NaN,NaN,NaN,NaN,NaN,NaN
producto,2187,1,BONUS 250 EC REGULADOR FISIOLÓGICO,2187,NaN,NaN,NaN,NaN,NaN,NaN,NaN
dosismad,2187.0,NaN,NaN,NaN,0.993278,0.309096,0.0,0.8,1.0,1.2,9.0
semsmad,2187.0,NaN,NaN,NaN,9.164838,3.441579,-1.6,7.1,8.7,10.6,45.0
edad,2187.0,NaN,NaN,NaN,12.766118,1.117866,10.3,12.0,12.5,13.3,21.1


## 0. Alcance del dataset

`BD_IPSA_1940` es un recorte del Ingenio Providencia, no una muestra aleatoria. Si una condición de
manejo (variedad, maduración o tipo de cosecha) es igual en todas las filas, su efecto no se puede
estimar y las conclusiones solo aplican a suertes que compartan esa condición.

En esta sección se verifica si `variedad`, `madurada`, `producto` y `tipocorte` tienen un único valor
en todo el dataset.

In [14]:
# 0. Verificar que las columnas de alcance son constantes
cols_alcance = ["variedad", "madurada", "producto", "tipocorte"]

resumen_alcance = pd.DataFrame({
    "n_valores_unicos": df_ipsa[cols_alcance].nunique(),
    "valor": [df_ipsa[c].iloc[0] for c in cols_alcance],
})
display(resumen_alcance)

# Detener la ejecución si alguna columna de alcance no es constante
assert (resumen_alcance["n_valores_unicos"] == 1).all(), "Alguna columna de alcance NO es constante"

,n_valores_unicos,valor
variedad,1,CC01-1940
madurada,1,SI
producto,1,BONUS 250 EC REGULADOR FISIOLÓGICO
tipocorte,1,Mecanizado Verde


### Conclusión

Las cuatro columnas tienen un único valor en las 2.187 filas:

| Columna | Valor único | Qué no se puede estudiar con este dataset |
|---|---|---|
| `variedad` | CC01-1940 | Diferencias entre variedades |
| `madurada` | SI | Suertes maduradas vs. no maduradas |
| `producto` | BONUS 250 EC | Diferencias entre madurantes |
| `tipocorte` | Mecanizado Verde | Cosecha mecanizada vs. manual o con quema |

- Las cuatro columnas se excluyen como predictores porque, al ser constantes, no aportan información.
- Los resultados de los modelos aplican solo a suertes de CC01-1940 maduradas químicamente y
  cosechadas en verde con máquina.

## 1. Diccionario de datos

El diccionario oficial de `BD_IPSA` describe cada columna de forma breve y, en algunos casos, genérica.
Antes de usar una columna como predictor hay que confirmar qué mide y en qué unidad, porque una
interpretación equivocada puede invalidar el análisis posterior.

En esta sección se compara la descripción oficial de cada columna con su comportamiento real en los
datos (tipo, número de valores distintos y rango). Las columnas cuya descripción falte, sea ambigua o
no sea coherente con los valores observados se investigan por separado.

In [15]:
# Descripciones del diccionario oficial de BD_IPSA ("" = sin descripción)
dicc_oficial = {
    "NOME": "Nombre del registro (finca, campo o unidad de análisis)",
    "FAZ": "Código o identificador de la hacienda o finca",
    "TAL": "Identificador de una subunidad o lote",
    "tipocorte": "Tipo de corte realizado",
    "variedad": "Variedad del cultivo",
    "madurada": "Indicador relacionado con la maduración del cultivo",
    "producto": "",
    "dosismad": "Dosis aplicada de fertilizantes o agroquímicos",
    "semsmad": "Dosis aplicada de fertilizantes o agroquímicos",
    "edad": "Edad del cultivo al momento del registro (días, semanas o meses)",
    "cortes": "Número de cortes o cosechas realizadas en el periodo",
    "me": "",
    "vejez": "Edad o antigüedad del cultivo",
    "sacarosa": "Porcentaje o cantidad de sacarosa",
    "mes": "Mes del registro o de la cosecha",
    "periodo": "Periodo o año del registro",
    "TCH": "Toneladas de caña por hectárea",
    "lluvias": "Lluvias registradas en el periodo (mm)",
    "grupo_tenencia": "Categoría de tenencia del terreno",
    "pct_diatrea": "Porcentaje de infestación por diatrea",
}

filas = []
for col in df_ipsa.columns:
    s = df_ipsa[col]
    es_num = pd.api.types.is_numeric_dtype(s)
    filas.append({
        "columna": col,
        "descripcion_oficial": dicc_oficial.get(col, ""),
        "tipo": str(s.dtype),
        "n_unicos": s.nunique(),
        "min": s.min() if es_num else None,
        "mediana": s.median() if es_num else None,
        "max": s.max() if es_num else None,
        "ejemplos": list(s.drop_duplicates().head(4)),
    })

inventario = pd.DataFrame(filas).set_index("columna")
display(inventario)

,descripcion_oficial,tipo,n_unicos,min,mediana,max,ejemplos
columna,,,,,,,
NOME,"Nombre del registro (finca, campo o unidad de ...",str,285,NaN,NaN,NaN,"[AMAIME SILCA, ARANJUEZ, AURORA CUCALON, BARCE..."
FAZ,Código o identificador de la hacienda o finca,int64,285,80100.0,80396.0,82519.0,"[81291, 80552, 80601, 80492]"
TAL,Identificador de una subunidad o lote,object,273,NaN,NaN,NaN,"[40, 41, 43, 1]"
tipocorte,Tipo de corte realizado,str,1,NaN,NaN,NaN,[Mecanizado Verde]
variedad,Variedad del cultivo,str,1,NaN,NaN,NaN,[CC01-1940]
madurada,Indicador relacionado con la maduración del cu...,str,1,NaN,NaN,NaN,[SI]
producto,,str,1,NaN,NaN,NaN,[BONUS 250 EC REGULADOR FISIOLÓGICO]
dosismad,Dosis aplicada de fertilizantes o agroquímicos,float64,18,0.0,1.0,9.0,"[0.8, 0.6, 1.2, 1.0]"
semsmad,Dosis aplicada de fertilizantes o agroquímicos,float64,150,-1.6,8.7,45.0,"[8.3, 6.3, 7.9, 6.6]"


### Observaciones sobre el inventario

**Columnas cuya descripción es coherente con los valores observados:**

- `NOME` y `FAZ` tienen 285 valores distintos cada una, lo que sugiere que son el nombre y el código de
  la misma hacienda. `TAL` tiene 273 valores y es de tipo `object`, es decir, mezcla tipos de dato.
- `edad` va de 10.3 a 21.1 (mediana 12.5). La descripción oficial no fija la unidad, pero ese rango solo
  es coherente con meses: un ciclo de cultivo dura aproximadamente entre 12 y 18 meses.
- `cortes` toma valores enteros de 1 a 14.
- `sacarosa` (9.2 a 16.0 %), `TCH` (6 a 249 t/ha), `pct_diatrea` (0.2 a 25.5 %) y `lluvias` (0 a 1.468 mm)
  tienen unidades explícitas y rangos plausibles. Sus valores extremos se revisan más adelante.
- `mes` va de 1 a 12, y `periodo` tiene formato AAAAMM (201407 a 202101).

**Columnas que requieren revisión:**

| Columna | Problema |
|---|---|
| `me` | No tiene descripción en el diccionario. Rango de 3.4 a 15.0. |
| `vejez` | La descripción oficial ("antigüedad del cultivo") no es coherente con un rango de 0.2 a 102.9 con mediana 2.6, y la edad del cultivo ya está en `edad`. |
| `dosismad` | Descripción genérica y sin unidad. Máximo de 9.0 frente a una mediana de 1.0, y mínimo de 0.0 aunque todas las suertes figuran como maduradas. |
| `semsmad` | La descripción es idéntica a la de `dosismad`. Su rango (−1.6 a 45.0) no es coherente con una dosis, y un valor negativo no es coherente con ninguna cantidad física. |
| `grupo_tenencia` | Tres códigos (1, 2, 3) sin significado documentado. |

Además, antes de interpretar las columnas hay que confirmar qué representa cada fila. Con 285 haciendas
y 2.187 filas, cada hacienda aparece en varios registros.

### 1.1 Unidad de observación

Se verifica si `NOME` y `FAZ` identifican la misma hacienda, qué tipos de valores contiene `TAL` y si
sus valores de texto son códigos o números almacenados como texto, si `mes` es redundante con
`periodo`, y si la combinación hacienda + suerte + periodo identifica una fila única.

In [16]:
# 1.1 Unidad de observación

# ¿NOME y FAZ identifican la misma hacienda?
print("FAZ con más de un NOME:", (df_ipsa.groupby("FAZ")["NOME"].nunique() > 1).sum())
print("NOME con más de un FAZ:", (df_ipsa.groupby("NOME")["FAZ"].nunique() > 1).sum())

# TAL es de tipo object: ¿qué tipos de valores contiene?
display(df_ipsa["TAL"].map(lambda v: type(v).__name__).value_counts().to_frame("n_filas"))

# ¿mes coincide con los dos últimos dígitos de periodo?
print("Filas donde mes != periodo % 100:", (df_ipsa["mes"] != df_ipsa["periodo"] % 100).sum())

# TAL como texto para poder agrupar sin errores por la mezcla de tipos
tal_txt = df_ipsa["TAL"].astype(str)

# ¿hacienda + suerte + periodo identifica una fila única?
dup = df_ipsa.assign(TAL=tal_txt).duplicated(subset=["FAZ", "TAL", "periodo"], keep=False)
print("Filas con FAZ + TAL + periodo repetido:", dup.sum())

# ¿Cuántas cosechas tiene cada suerte en el dataset?
cosechas_por_suerte = df_ipsa.assign(TAL=tal_txt).groupby(["FAZ", "TAL"]).size()
print("Suertes distintas (FAZ + TAL):", len(cosechas_por_suerte))
display(cosechas_por_suerte.value_counts().sort_index().rename("n_suertes").to_frame())

FAZ con más de un NOME: 0
NOME con más de un FAZ: 0


,n_filas
TAL,
int,1598
str,589


Filas donde mes != periodo % 100: 0
Filas con FAZ + TAL + periodo repetido: 0
Suertes distintas (FAZ + TAL): 1115


,n_suertes
1,486
2,337
3,168
4,98
5,25
6,1


In [17]:
# Valores de TAL almacenados como texto: ¿son códigos alfanuméricos o números guardados como texto?
tal_str = df_ipsa.loc[df_ipsa["TAL"].map(lambda v: isinstance(v, str)), "TAL"]

es_numerico = tal_str.str.strip().str.fullmatch(r"\d+")
print("Valores de texto que son solo dígitos:", es_numerico.sum(), "de", len(tal_str))
print("Ejemplos:", tal_str.drop_duplicates().head(15).tolist())

Valores de texto que son solo dígitos: 0 de 589
Ejemplos: ['990A', '991B', '991C', '991D', '991E', '991G', '320B', '322A', '324A', '005A', '006A', '010A', '004A', '001A', '002B']


#### Conclusión de 1.1

- `NOME` y `FAZ` están en relación 1 a 1: son el nombre y el código de la misma hacienda, así que
  `NOME` es redundante con `FAZ`.
- `TAL` combina 1.598 valores enteros y 589 códigos alfanuméricos (por ejemplo, `005A` y `991B`).
  Ninguno de los valores de texto es un número almacenado como texto, por lo que la mezcla de tipos no
  duplica suertes. La letra final parece indicar una subdivisión de la suerte. Para agrupar, `TAL` se
  trata como texto.
- `mes` coincide en todas las filas con los dos últimos dígitos de `periodo`, así que su información ya
  está contenida en `periodo`.
- La combinación `FAZ` + `TAL` + `periodo` no se repite, así que **cada fila es una cosecha de una
  suerte**.
- El dataset contiene 1.115 suertes distintas. 486 aparecen en una sola cosecha y 629 en dos o más
  (hasta seis):

| Cosechas por suerte | 1 | 2 | 3 | 4 | 5 | 6 |
|---|---|---|---|---|---|---|
| Número de suertes | 486 | 337 | 168 | 98 | 25 | 1 |

Como una misma suerte aparece en varias filas, las observaciones no son independientes entre sí. Una
partición aleatoria de filas en entrenamiento y prueba puede dejar cosechas de la misma suerte en
ambos conjuntos, y el desempeño en prueba resultaría optimista. Esto se tiene en cuenta al construir
la partición en `02_modelos_clasificacion.ipynb`.